Step 1: import Problem class from UQPyL **problem** module

<p align="center"><img src="../docs/pic/Problem1.svg" width=500/></p>

In [2]:
from UQPyL.problems import Problem

Step 2: define objFunc Function  

The 'objFunc' function is one that accepts a 2D numpy array 'X' as input and returns a 2D numpy array 'objs'. Specifically:  
- The rows of the 2D array 'X' represent a set of decision variables, while the columns correspond to different values of the same variable.  
- The returned 2D array 'objs' should have the same number of rows as 'X', and the number of columns should equal the number of objectives in the problem.  

For a single-objective problem, the shape of the 2D array 'objs' should be (N, 1), where 'N' is the number of input variables.  
For a multi-objective problem, the shape of the 2D array 'objs' should be (N, M), where 'M' is the number of objective functions. 

Users should ensure that the returned 2D array 'objs' satisfies the above shape requirements.  

In [3]:
def objFunc(X):
    # If possible, advise vectorizing operations on matrix X to improve computational efficiency.
    objs =100 * (X[:, 2] - X[:, 1]**2)**2+ 100 * (X[:, 1] - X[:, 0]**2)**2  + \
            (1 - X[:, 1])**2 + (1 - X[:, 0])**2 
    return objs[:, None] # Although UQPyL performs further checks, please ensure the returned `objs` is a 2D array.

UQPyL also supports an alternative way of defining the objFunc function.  
For problems involving numerical simulation models, it's often not feasible to vectorize operations on matrix 'X'.  
To address this, UQPyL provides a decorator function @singleFunc that enables single running mode.  
In single running mode, The 'objFunc' function only accepts a Python list or 1D numpy array as input.  
It processes one decision variable combination at a time, making it suitable for complex or non-vectorizable objective functions.  

In [4]:
# First, import the decorator that enables singleton mode from UQPyL
from UQPyL.problems import singleFunc

In [5]:
@singleFunc
def objFunc_(X):  # Input X should be a 1D numpy array or Python list
     # Perform calculations for each element in X
    obj = 100 * (X[2] - X[1]**2)**2 + 100 * (X[1] - X[0]**2)**2 + \
            (1 - X[1])**2 + (1 - X[0])**2 
    return obj # Return the objective value: a scalar for single-objective, or a 1D array/list for multi-objective

Step 3: Define concFunc Function  
Similar to objFunc, the conFunc function supports two definition modes.  
Note: The return value of conFunc indicates whether the constraints are satisfied:
  - A negative value indicates a violation of the constraint — the smaller the value, the more severe the violation.  
  - A positive value indicates the constraint is satisfied, i.e., the solution is feasible.  
As a result, users may need to reformulate their original constraint expressions to follow this convention.  

In [6]:
# Matrix Mode
def conFunc(X):
    cons = X[:, 0]**2 + X[:, 1]**2 + X[:, 2]**2 - 4 
    return cons[:, None] 

In [7]:
# Single Running Mode
@singleFunc
def conFunc(X):
    con = X[0]**2 + X[1]**2 + X[2]**2 - 4 
    return con

Step 4: describe the properties of X

In [8]:
nInput = 3 # number of input variables (X), here it's 3 inputs.
nOutput = 1 # number of outputs (objective functions), here it's 1 objective.

#Upper bound of X.
ub = [10, 10, 10] # It can be a float, int, list, or numpy array. 
# In this case, both input variables have an upper bound of 10. 

# Lower bound of X.
lb = [0, 0, 0] # It can also be a float, int, list, or numpy array. 
# In this case, both input variables have a lower bound of 0.

# Types of variables.
# type 0 for continuous, 1 for integer, and 2 for discrete.
varType = [0, 1, 2]  
# varType[0] = 0: The first input (X[0]) is a float.
# varType[1] = 1: The second input (X[1]) is an integer variable.
# varType[2] = 2: The second input (X[2]) is a discrete variable.

# The set of possible values for discrete variables.
varSet = {2: [2, 3.4, 5.1, 7]} 
# varSet is a dictionary where the key indicates the index of the variable (2 refers to the third variable, x3). It follows Python's zero-based indexing.
# The value associated with key 2 specifies the set of possible values for X[2]: [2, 3.4, 5.1, 7].
# This means that X[2] can only take one of these four values: 2, 3.4, 5.1, or 7.

# The optimization type: 'min' for minimization, 'max' for maximization.
optType = 'min'

# Names (or labels) for the input variables.
xLabels = ['x1', 'x2', 'x3'] 
# If the optimization problem has named variables, you can set them here.
# Otherwise, default names like 'x1', 'x2', 'x3', etc., can be used.

# Names (or labels) for the objective functions.
yLabels = ['obj1'] # Similar to xLabel, if your objective(s) have specific names, you can set them here.
# Otherwise, use default labels like 'obj1', 'obj2', etc.

# Name of the optimization problem
name = 'Rosenbrock'
# Useful for identifying the problem instance, organizing results, saving files, etc.

Step 5: Initialize the problem instance

In [9]:
problem = Problem(
    nInput=nInput,
    nOutput=nOutput,
    objFunc=objFunc,
    conFunc=conFunc,
    ub=ub,
    lb=lb,
    varType=varType,
    varSet=varSet,
    xLabels=xLabels,
    yLabels=yLabels,
    name=name
)

Step 6: Import optimization methods from UQPyL  
All methods and algorithms in UQPyL operate by reading the 'problem' object  
In this example, we are using the Genetic Algorithm (GA) for optimization  

In [10]:
from UQPyL.optimization.single_objective import GA

Create an instance of the Genetic Algorithm (GA).  
By default, GA will output optimization history and final results in the command line.

In [13]:
ga = GA(verboseFreq = 100, saveFlag = False)

Run the Genetic Algorithm optimization by passing the defined 'problem' object

In [14]:
ga.run(problem = problem)

======================================================GA Setting======================================================
+------+------+------+------+------+
| proC | disC | proM | disM | nPop |
+------+------+------+------+------+
|  1   |  20  |  1   |  20  |  50  |
+------+------+------+------+------+

==================================================FEs: 50 | Iters: 0==================================================
+-----------------+-----------------+-----------------+-----------------+
|       FEs       |      Iters      |     OptType     |     Feasible    |
+-----------------+-----------------+-----------------+-----------------+
|        50       |        0        |       min       |      False      |
+-----------------+-----------------+-----------------+-----------------+

+-----------------+-----------------+-----------------+-----------------+
|       obj1      |        x1       |        x2       |        x3       |
+-----------------+-----------------+-----------------+--